GRADIENT BOOSTING NO PCA

Jack Silkaitis

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import mean_squared_error
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.decomposition import PCA

In [2]:
# Import data
df = pd.read_csv('./CleanPreprocessedExoplanet.csv')
df.head(3)
df.shape

(15351, 51)

In [3]:
df.columns

Index(['Unnamed: 0', 'sy_snum', 'sy_pnum', 'cb_flag', 'pl_controv_flag',
       'ttv_flag', 'st_nphot', 'pl_nespec', 'glon', 'glat', 'dec', 'ra',
       'elon', 'elat', 'release_month', 'st_nrvc', 'pl_nnotes', 'release_year',
       'pl_ndispec', 'pl_ntranspec', 'st_nspec', 'disc_year', 'sy_vmag',
       'sy_tmag', 'sy_hmag', 'sy_kmag', 'sy_jmag', 'sy_pm', 'sy_pmra',
       'sy_pmdec', 'sy_gaiamag', 'sy_bmag', 'st_rad', 'sy_dist', 'sy_w1mag',
       'sy_w2mag', 'sy_w3mag', 'sy_w4mag', 'sy_plx', 'st_teff', 'pl_orbper',
       'sy_kepmag', 'st_mass', 'soltype_Kepler Project Candidate (q1_q12_koi)',
       'soltype_Kepler Project Candidate (q1_q16_koi)',
       'soltype_Kepler Project Candidate (q1_q17_dr24_koi)',
       'soltype_Kepler Project Candidate (q1_q17_dr25_koi)',
       'soltype_Kepler Project Candidate (q1_q8_koi)',
       'soltype_Published Candidate', 'soltype_Published Confirmed',
       'log radius'],
      dtype='object')

In [4]:
# Split and prepare data
(df_train,df_test) = train_test_split(df,train_size=0.8,
                                      test_size=0.2,
                                      random_state=0)

In [10]:
X_train = df_train.drop(['log radius','Unnamed: 0'],axis=1)
y_train = df_train['log radius']
X_test = df_test.drop(['log radius','Unnamed: 0'],axis=1)
y_test = df_test['log radius']

In [12]:
stnd = StandardScaler().set_output(transform='pandas')

X_number = X_train.select_dtypes(include='number')
X_categorical = X_train.select_dtypes(exclude='number')
X_number = stnd.fit_transform(X_number)
X_train = pd.concat([X_number, X_categorical], axis=1)

X_number = X_test.select_dtypes(include='number')
X_categorical = X_test.select_dtypes(exclude='number')
X_number = stnd.transform(X_number)
X_test = pd.concat([X_number, X_categorical], axis=1)

In [13]:
# Baseline before applying PCA, arbitrary max_depth
tree = DecisionTreeRegressor(max_depth=2)
tree.fit(X_train,y_train)
tree.score(X_test, y_test)

0.09993551493313568

In [ ]:
B = np.arange(1,301,100)
C = np.arange(1,10,3)
D = np.arange(0.01,1,0.33)
grid = {'n_estimators':B, 'max_depth':C, 'learning_rate':D}

gbt = GradientBoostingRegressor()
gbtCV = GridSearchCV(gbt,param_grid=grid,return_train_score=True,n_jobs=-1, verbose=2)
gbtCV.fit(X_train,y_train)

print('best params =',gbtCV.best_params_, '  valid acc =',gbtCV.best_score_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
best params = {'learning_rate': np.float64(0.34), 'max_depth': np.int64(7), 'n_estimators': np.int64(201)}   valid acc = 0.8628368954556851


In [16]:
B = np.arange(150,300,50)
C = np.arange(6,9,1)
D = np.arange(.2, .5, .1)
grid = {'n_estimators':B, 'max_depth':C, 'learning_rate':D}

gbt = GradientBoostingRegressor()
gbtCV = GridSearchCV(gbt,param_grid=grid,return_train_score=True,n_jobs=-1, verbose=2)
gbtCV.fit(X_train,y_train)

print('best params =',gbtCV.best_params_, '  valid acc =',gbtCV.best_score_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
best params = {'learning_rate': np.float64(0.30000000000000004), 'max_depth': np.int64(8), 'n_estimators': np.int64(200)}   valid acc = 0.879379417875023


In [17]:
B = np.arange(175,250,25)
C = np.arange(.1,.3,.05)
grid = {'n_estimators':B, 'learning_rate':C}

gbt = GradientBoostingRegressor(max_depth=8)
gbtCV = GridSearchCV(gbt,param_grid=grid,return_train_score=True,n_jobs=-1, verbose=2)
gbtCV.fit(X_train,y_train)

print('best params =',gbtCV.best_params_, '  valid acc =',gbtCV.best_score_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
best params = {'learning_rate': np.float64(0.15000000000000002), 'n_estimators': np.int64(200)}   valid acc = 0.8813205651428502


In [18]:
print('Test R2',gbtCV.score(X_test,y_test))
print('Test R2',gbtCV.score(X_train,y_train))

Test R2 0.93049053121004
Test R2 0.9950967422677197


In [19]:
y_pred = gbtCV.predict(X_test)
mean_squared_error(y_pred, y_test)

0.005555372095173796

In [21]:
y_pred = gbtCV.predict(X_train)
mean_squared_error(y_pred, y_train)

0.00041121893703016893